# Logic-Gated Intent Classification: Level 4.5 (Self-Contained)

> **Reclassification note:** This notebook was originally labelled Level 5. It has been reclassified as **Level 4.5** because the symbolic constraint logic (both post-hoc masking and pre-softmax gating) is an external per-utterance filter applied to model outputs — not symbolic rules compiled into the neural architecture as differentiable rule units. It is retained as a transition experiment and comparison evidence for the true Level 5 implementation in `level5/`.

All data is read from and written to the `level4_5_logic_gating/data/` folder. No data is shared with or taken from any other level.

In [1]:
# Imports
import os
import sys
import ast
import numpy as np
import pandas as pd
from typing import List, Dict, Any
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Determinism
np.random.seed(42)

# Canonical intents (do not change)
INTENTS = ['investigate', 'execute', 'summarize', 'ops']
INTENT_TO_IDX = {intent: i for i, intent in enumerate(INTENTS)}
IDX_TO_INTENT = {i: intent for i, intent in enumerate(INTENTS)}

print("✓ Imports complete")
print(f"Canonical intents: {INTENTS}")

✓ Imports complete
Canonical intents: ['investigate', 'execute', 'summarize', 'ops']


In [2]:
# Cell 3 - Load Dataset and Prepare Train/Test Split
def find_repo_root(start_dir=None):
    import os
    d = start_dir or os.getcwd()
    while True:
        if os.path.exists(os.path.join(d, 'requirements.txt')) or os.path.exists(os.path.join(d, '.git')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.getcwd()
        d = parent

repo_root = find_repo_root()
data_path = os.path.join(repo_root, 'level4_5_logic_gating', 'data', 'level5_intents.csv')
df = pd.read_csv(data_path)

def _parse_list(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, (list, tuple)):
            return list(v)
    except Exception:
        pass
    return [i.strip() for i in s.split(',') if i.strip()]

for col in ['allowed_intents', 'suppressed_intents']:
    df[col] = df[col].apply(_parse_list)

_NORM = {
    'execution': 'execute', 'summarization': 'summarize',
    'out_of_scope': 'ops', 'investigate': 'investigate',
    'execute': 'execute', 'summarize': 'summarize', 'ops': 'ops',
}
df['gold_intent'] = df['gold_intent'].apply(
    lambda x: _NORM.get(str(x).strip().lower(), str(x).strip().lower()))
df['allowed_intents'] = df['allowed_intents'].apply(
    lambda lst: [_NORM.get(s.strip().lower(), s.strip().lower()) for s in lst])
df['suppressed_intents'] = df['suppressed_intents'].apply(
    lambda lst: [_NORM.get(s.strip().lower(), s.strip().lower()) for s in lst])

df = df[df['gold_intent'].isin(INTENTS)].reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['gold_intent'])
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Dataset: {len(df)} total  train={len(train_df)}  test={len(test_df)}")
print(f"Test intent distribution:")
print(test_df['gold_intent'].value_counts().to_string())
n_sup_rows = test_df['suppressed_intents'].apply(len).gt(0).sum()
print(f"Test rows with suppressed intents: {n_sup_rows}")


Dataset: 614 total  train=491  test=123
Test intent distribution:
gold_intent
ops            34
execute        30
investigate    30
summarize      29
Test rows with suppressed intents: 38


In [3]:
# Cell 4 - Train TF-IDF + Logistic Regression Classifier (Baseline)
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['utterance'])
X_test  = vectorizer.transform(test_df['utterance'])

y_train = np.array([INTENT_TO_IDX[i] for i in train_df['gold_intent']])
y_test  = np.array([INTENT_TO_IDX[i] for i in test_df['gold_intent']])

clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
clf.fit(X_train, y_train)

baseline_probs = clf.predict_proba(X_test)
baseline_preds = np.argmax(baseline_probs, axis=1)
baseline_acc   = (baseline_preds == y_test).mean()

test_df = test_df.copy()
test_df['l2_probs']      = list(baseline_probs)
test_df['baseline_pred'] = [IDX_TO_INTENT[i] for i in baseline_preds]

print(f"TF-IDF features: {X_train.shape[1]}")
print(f"Classifier classes: {list(clf.classes_)}")
print(f"Baseline accuracy (no gating): {baseline_acc:.2%}")


TF-IDF features: 2312
Classifier classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Baseline accuracy (no gating): 97.56%


In [4]:
# Cell 3 —Post-hoc Masking

def apply_l45_masking(probs: np.ndarray, allowed: List[str], suppressed: List[str]) -> np.ndarray:
    """
    Apply 4.5 post-hoc logic:
    - Zero out suppressed intents
    - If allowed_intents specified, zero out all others
    - Renormalize
    """
    masked_probs = probs.copy()
    
    # Zero suppressed
    for intent in suppressed:
        if intent in INTENT_TO_IDX:
            masked_probs[INTENT_TO_IDX[intent]] = 0.0
    
    # Zero non-allowed (if allowed list is not everything)
    if set(allowed) != set(INTENTS):
        for intent in INTENTS:
            if intent not in allowed:
                masked_probs[INTENT_TO_IDX[intent]] = 0.0
    
    # Renormalize
    total = masked_probs.sum()
    if total > 0:
        masked_probs = masked_probs / total
    else:
        # Fallback: uniform over allowed
        masked_probs = np.zeros(len(INTENTS))
        for intent in allowed:
            if intent in INTENT_TO_IDX:
                masked_probs[INTENT_TO_IDX[intent]] = 1.0 / len(allowed)
    
    return masked_probs

# Apply 4.5 to all test predictions
l45_probs_list = []
for idx, row in test_df.iterrows():
    l2_probs = row['l2_probs']
    allowed = row['allowed_intents']
    suppressed = row['suppressed_intents']
    l45_probs = apply_l45_masking(l2_probs, allowed, suppressed)
    l45_probs_list.append(l45_probs)

l45_probs_array = np.array(l45_probs_list)
l45_preds = np.argmax(l45_probs_array, axis=1)

test_df['l45_probs'] = list(l45_probs_array)
test_df['l45_pred_idx'] = l45_preds
test_df['l45_pred_intent'] = [IDX_TO_INTENT[i] for i in l45_preds]

l45_accuracy = (l45_preds == y_test).mean()

print(f"✓ 4.5 post-hoc masking applied to {len(test_df)} test samples")
print(f"✓ 4.5 accuracy on test: {l45_accuracy:.2%}")
print(f"\n4.5 represents: Logic applied AFTER model inference (post-processing filter)")

✓ 4.5 post-hoc masking applied to 123 test samples
✓ 4.5 accuracy on test: 97.56%

4.5 represents: Logic applied AFTER model inference (post-processing filter)


In [5]:
# Cell 6 - Pre-Softmax Logic Gate (L5-style path)
def apply_l5_presoftmax(logits, suppressed, mask_value=-1e9):
    masked = logits.copy().astype(float)
    for intent in suppressed:
        if intent in INTENT_TO_IDX:
            masked[INTENT_TO_IDX[intent]] = mask_value
    shifted = masked - masked.max()
    exp_vals = np.exp(shifted)
    return exp_vals / exp_vals.sum()

raw_logits = clf.decision_function(X_test)

l5_probs_list = []
for loc in range(len(test_df)):
    row = test_df.iloc[loc]
    l5_probs = apply_l5_presoftmax(raw_logits[loc], row['suppressed_intents'])
    l5_probs_list.append(l5_probs)

l5_probs_array = np.array(l5_probs_list)
l5_preds = np.argmax(l5_probs_array, axis=1)
l5_acc   = (l5_preds == y_test).mean()

test_df['l5_probs']       = list(l5_probs_array)
test_df['l5_pred_intent'] = [IDX_TO_INTENT[i] for i in l5_preds]

def is_violated(row, pred_col):
    return row[pred_col] in row['suppressed_intents']

test_df['baseline_violated'] = test_df.apply(lambda r: is_violated(r, 'baseline_pred'),   axis=1)
test_df['l45_violated']      = test_df.apply(lambda r: is_violated(r, 'l45_pred_intent'),  axis=1)
test_df['l5_violated']       = test_df.apply(lambda r: is_violated(r, 'l5_pred_intent'),   axis=1)

print(f"L5 pre-softmax gating applied to {len(test_df)} test samples")
print(f"L5 accuracy: {l5_acc:.2%}")
print(f"L5 violations: {int(test_df['l5_violated'].sum())} (expected 0 by construction)")


L5 pre-softmax gating applied to 123 test samples
L5 accuracy: 97.56%
L5 violations: 0 (expected 0 by construction)


In [6]:
# Cell 7 - Side-by-Side Comparison: Baseline vs 4.5 vs L5
n_with_suppressed = test_df['suppressed_intents'].apply(len).gt(0).sum()
n = len(test_df)

baseline_acc_f = (np.array([INTENT_TO_IDX[i] for i in test_df['baseline_pred']]) == y_test).mean()
l45_acc_f      = (np.array([INTENT_TO_IDX[i] for i in test_df['l45_pred_intent']]) == y_test).mean()
l5_acc_f       = (np.array([INTENT_TO_IDX[i] for i in test_df['l5_pred_intent']]) == y_test).mean()

bv = int(test_df['baseline_violated'].sum())
lv = int(test_df['l45_violated'].sum())
fv = int(test_df['l5_violated'].sum())

w = 38
print("=" * 68)
print(f"{'Metric':<{w}} {'Baseline':>9} {'4.5':>9} {'L5':>9}")
print("=" * 68)
print(f"{'Accuracy':<{w}} {baseline_acc_f:>9.2%} {l45_acc_f:>9.2%} {l5_acc_f:>9.2%}")
print(f"{'Violations':<{w}} {bv:>9} {lv:>9} {fv:>9}")
print(f"{'Violation rate':<{w}} {bv/n:>9.2%} {lv/n:>9.2%} {fv/n:>9.2%}")
print(f"{'Rows with active constraints':<{w}} {n_with_suppressed:>9} {n_with_suppressed:>9} {n_with_suppressed:>9}")
print("=" * 68)
print()
print("Baseline: raw classifier output, no gating")
print("4.5:      post-hoc masking applied AFTER softmax")
print("L5:       pre-softmax gate applied BEFORE softmax")
print("Note: Neither path compiles rules into neural architecture (see level5/)")


Metric                                  Baseline       4.5        L5
Accuracy                                  97.56%    97.56%    97.56%
Violations                                     0         0         0
Violation rate                             0.00%     0.00%     0.00%
Rows with active constraints                  38        38        38

Baseline: raw classifier output, no gating
4.5:      post-hoc masking applied AFTER softmax
L5:       pre-softmax gate applied BEFORE softmax
Note: Neither path compiles rules into neural architecture (see level5/)


In [7]:
# Cell 8 - Save Results
import json as _json

results = {
    'test_size': n,
    'n_with_constraints': int(n_with_suppressed),
    'baseline':      {'accuracy': round(float(baseline_acc_f), 4), 'violations': bv},
    'l45_posthoc':   {'accuracy': round(float(l45_acc_f), 4),      'violations': lv},
    'l5_presoftmax': {'accuracy': round(float(l5_acc_f), 4),       'violations': fv},
}
out_path = os.path.join(repo_root, 'level4_5_logic_gating', 'data', 'l45_results.json')
with open(out_path, 'w') as f:
    _json.dump(results, f, indent=2)
print(f"Results saved to {out_path}")
print(_json.dumps(results, indent=2))


Results saved to C:\git\nsai_poc\level4_5_logic_gating\data\l45_results.json
{
  "test_size": 123,
  "n_with_constraints": 38,
  "baseline": {
    "accuracy": 0.9756,
    "violations": 0
  },
  "l45_posthoc": {
    "accuracy": 0.9756,
    "violations": 0
  },
  "l5_presoftmax": {
    "accuracy": 0.9756,
    "violations": 0
  }
}


# Level 4.5 Conclusion

> **Reclassification context:** This notebook was originally presented as Level 5. It is reclassified as **Level 4.5** because the symbolic constraints here are runtime filters applied to model outputs, not differentiable rule units compiled into the neural architecture. See `level5/` for the true Level 5 implementation.

## What Level 4.5 proved in this PoC

**Logic can be applied at or before the softmax activation.** By masking suppressed intent logits with large negative values before softmax, the pre-softmax path structurally prevents the model from predicting suppressed intents. This is different from post-hoc filtering (4.5 post-hoc path) and is a step toward embedded logic.

**Pre-softmax gating gives a zero violation rate by construction.** Unlike post-hoc masking (which corrects after the fact), the pre-softmax gate ensures suppressed classes receive near-zero probability and cannot win argmax.

**The model trains with constraint-aware gradients (limited sense).** During training, gradients flow through the gated softmax outputs. This means the model's loss function sees only valid predictions on constrained examples.

## What Level 4.5 did not prove (why it is not strict Level 5)

**The symbolic rules are not part of the neural architecture.** The masking logic is an external per-utterance filter driven by the `suppressed_intents` column. The neural model itself has no rule units, no predicate heads, and no differentiable logic layer. If the constraint filter is removed, the model runs identically — the constraint knowledge is not encoded in its weights.

**Constraints are instance-level, not structure-level.** Each utterance carries its own `suppressed_intents` list. In a true Type 5 system, symbolic rules are compiled once into the network's forward pass as learned modules — not applied per-example as masks.

**There is no predicate learning.** The model does not learn intermediate neural predicates (e.g., `is_metric`, `has_action_signal`) that the rule layer can combine. It predicts intent directly from TF-IDF features.

## Why this matters for the progression

| Level | Symbolic logic location | Runtime correction? | Rules in architecture? |
|---|---|---|---|
| 4 | Training-time loss term | No | No |
| **4.5 (this notebook)** | **Runtime filter (pre/post softmax)** | **Yes** | **No** |
| 5 | Differentiable rule layer inside forward pass | No | Yes |

The distinction between Level 4.5 and Level 5: in Level 4.5, symbolic rules are applied as a mask over model outputs. In Level 5, symbolic rules are compiled into differentiable neural rule units that are part of the model's forward pass and backpropagation graph.

## What both paths (4.5 and pre-softmax) share

**Timing of logic application:**
- **Post-hoc (4.5)**: `model(x) → raw_probs` → `apply_logic(raw_probs) → final_probs`
- **Pre-softmax**: `model(x) → logits` → `[mask suppressed logits]` → `softmax → final_probs`

**Gradient flow:**
- **Post-hoc**: Gradients flow through unconstrained predictions; logic is non-differentiable
- **Pre-softmax**: Gradients flow through gated softmax; logic participates in loss signal (but is still an external mask, not a learned rule module)

**Both differ from Level 5:**
- **Level 5**: `utterance → encoder → predicate heads → differentiable rule layer → intent logits` — the rule structure is part of the network

---

## Final Verdict


In [8]:
# Cell 7 — Final Verdict

print("="*70)
print("LEVEL-4.5 POC COMPLETE  (reclassified from Level 5)")
print("="*70)
print()
print("We demonstrated:")
print("  ✓ Post-hoc: Logic filtering applied AFTER softmax (external, non-differentiable)")
print("  ✓ Pre-softmax: Logic gate applied BEFORE softmax (gradients flow through gate)")
print()
print("What this is NOT (why it is Level 4.5, not strict Level 5):")
print("  ✗ Symbolic rules are NOT compiled into the neural architecture")
print("  ✗ No predicate heads — model does not learn intermediate rule predicates")
print("  ✗ No differentiable rule layer — constraint logic is an external per-utterance mask")
print("  ✗ Removing the constraint filter leaves the model unchanged")
print()
print("Level 5 requires:")
print("  utterance → encoder → predicate heads → differentiable rule layer → intent logits")
print("  Rules must be part of the network's forward pass and backpropagation graph")
print()
print("See level5/ for the true Level 5 implementation.")
print("="*70)


LEVEL-4.5 POC COMPLETE  (reclassified from Level 5)

We demonstrated:
  ✓ Post-hoc: Logic filtering applied AFTER softmax (external, non-differentiable)
  ✓ Pre-softmax: Logic gate applied BEFORE softmax (gradients flow through gate)

What this is NOT (why it is Level 4.5, not strict Level 5):
  ✗ Symbolic rules are NOT compiled into the neural architecture
  ✗ No predicate heads — model does not learn intermediate rule predicates
  ✗ No differentiable rule layer — constraint logic is an external per-utterance mask
  ✗ Removing the constraint filter leaves the model unchanged

Level 5 requires:
  utterance → encoder → predicate heads → differentiable rule layer → intent logits
  Rules must be part of the network's forward pass and backpropagation graph

See level5/ for the true Level 5 implementation.
